# Intrusion Detection with Softmax Regression
In this laboratory, we will use Softmax Regression to classy the network traffic flows as benign or malicious. The Logistic regression model returns a value between 0 and 1, which is the probability of the input flow of being malicious. We use a threshold set to 0.5 to decide whether the network flow is malicious or not.
We will train a logistic regression model on a dataset of benign traffic and DDoS attack traffic.

We will use a dataset of benign and various DDoS attacks from the CIC-DDoS2019 dataset (https://www.unb.ca/cic/datasets/ddos-2019.html).
The network traffic has been previously pre-processed in a way that packets are grouped in bi-directional traffic flows using the 5-tuple (source IP, destination IP, source Port, destination Port, protocol). Each flow is represented with 21 packet-header features computed from max 10 packets:

| Features           | Softmax Regression model           |
|---------------------|--------------------|
| timestamp (mean IAT)  <br> packet_length (mean) <br> IP_flags_df (sum) <br> IP_flags_mf (sum) <br> IP_flags_rb (sum) <br> IP_frag_off (sum) <br> protocols (mean) <br> TCP_length (mean) <br> TCP_flags_ack (sum) <br> TCP_flags_cwr (sum) <br> TCP_flags_ece (sum) <br> TCP_flags_fin (sum) <br> TCP_flags_push (sum) <br> TCP_flags_res (sum) <br> TCP_flags_reset (sum) <br> TCP_flags_syn (sum) <br> TCP_flags_urg (sum) <br> TCP_window_size (mean) <br> UDP_length (mean) <br> ICMP_type (mean) <br> Packets (counter) <br>| <img src="./softmax_regression_CIC2019.png" width="100%">  |

In [1]:
# Author: Roberto Doriguzzi-Corin
# Project: Course on Network Intrusion and Anomaly Detection with Machine Learning
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Import necessary libraries

import numpy as np
import glob
import h5py
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from keras.models import Sequential
from keras.layers import Dense
from tensorflow.keras.optimizers import SGD
from util_functions import *
DATASET_FOLDER = "./DOS2019"

In [2]:
# Load training, validation and test sets
feature_names = get_feature_names()
target_names = ['benign', 'dns',  'syn', 'udplag', 'webddos'] 
target_names_full = ['benign', 'dns', 'ldap', 'mssql', 'netbios', 'ntp', 'portmap', 'snmp', 'ssdp', 'syn', 'tftp', 'udp', 'udplag', 'webddos']
X_train, y_train = load_dataset(DATASET_FOLDER + "/*" + '-train.hdf5')
X_val, y_val = load_dataset(DATASET_FOLDER + "/*" + '-val.hdf5')
X_test, y_test = load_dataset(DATASET_FOLDER + "/*" + '-test.hdf5')

## Model definition

In [3]:
# Softmax Regression model
def SoftmaxRegression(model_name, input_shape,classes):
    ### define the activation function
    activation_function = ''
    model = Sequential(name=model_name)
    model.add(Dense(classes, input_shape=input_shape,activation=activation_function, name='fc1'))

    print(model.summary())
    return model

## Cost function and optimisation algorithm

In [4]:
def compileModel(model,lr):
    ### define the loss function
    loss_function = ''
    optimizer = SGD(learning_rate=lr, momentum=0.0) # the optimisation algorithm
    model.compile(loss=loss_function, optimizer=optimizer,metrics=['accuracy'])  # here we specify the loss function

## Train the model

In [6]:
model = SoftmaxRegression('log_reg', X_train.shape[1:4],len(target_names_full))
compileModel(model,0.001)

# Train the model
model.fit(X_train, y_train, epochs=500, batch_size=32, validation_data=(X_val, y_val))

Model: "log_reg"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 fc1 (Dense)                 (None, 14)                308       
                                                                 
Total params: 308
Trainable params: 308
Non-trainable params: 0
_________________________________________________________________
None
Epoch 1/500
26/26 [==============================] - 0s 3ms/step - loss: 5430.7559 - accuracy: 0.4933 - val_loss: 1996.0532 - val_accuracy: 0.3956
Epoch 2/500
26/26 [==============================] - 0s 879us/step - loss: 5542.2061 - accuracy: 0.5104 - val_loss: 5596.2427 - val_accuracy: 0.6154
Epoch 3/500
26/26 [==============================] - 0s 825us/step - loss: 6020.8555 - accuracy: 0.5594 - val_loss: 1565.2693 - val_accuracy: 0.5934
Epoch 4/500
26/26 [==============================] - 0s 752us/step - loss: 6117.1812 - accuracy: 0.5643 - val_loss: 4855.1787 - val_accurac

## Make prediction on unseen data

In [7]:
### make the prediction on the test set
y_pred =  
# Convert one-hot encoded predictions back to categorical values
y_pred_labels = np.argmax(y_pred, axis=1)
y_test_labels = np.argmax(y_test, axis=1)

### print the classification report
print()

              precision    recall  f1-score   support

      benign       0.31      0.57      0.40         7
         dns       1.00      0.87      0.93        15
         syn       0.96      1.00      0.98        23
      udplag       0.88      1.00      0.94        23
     webddos       0.96      0.74      0.83        34

    accuracy                           0.86       102
   macro avg       0.82      0.83      0.82       102
weighted avg       0.90      0.86      0.87       102

